# Correlation and Fringe Fitting from the Ground Up [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rndsrc/2026_eht-workshop/blob/main/tutorial.ipynb)

This notebook builds a simple polarized VLBI experiment from the ground up.

We begin with electric fields, define the Stokes parameters, and construct a polarized sky model.
We then show how telescopes with linear or circular feeds sample that radiation, leading naturally to the visibility function and the van Cittert–Zernike theorem: interferometric cross-correlations are Fourier samples of the sky.
We then introduce the main complications in real VLBI observations: telescope sensitivity, thermal noise, station gains, instrumental leakage, clock offsets, and atmospheric phase errors.
Finally, we show what fringe fitting is and how it is used to calibrate VLBI data.

## 0. Setup

We use `NumPy` for array calculations, `Astropy` for physical units and constants, and `Matplotlib` for plots.

In [ ]:
import numpy as np

from astropy import units     as u
from astropy import constants as c

from matplotlib import pyplot as plt

In [ ]:
nu0  = 230.0 * u.GHz # reference observing frequency
dnu  =   2.0 * u.GHz # total observing bandwidth
tint =  10.0 * u.s   # integration time
etaq =   0.88        # 2-bit quantization efficiency

In [ ]:
# Astropy `equivalencies` help us find the corresponding observing wavelength:

nu0.to(u.mm, equivalencies=u.spectral())

In [ ]:
# We should always seed the random number generator to make sure the results are reproducible.

np.random.seed(0)
X = np.random.normal(size=1024)

In [ ]:
print(X[:10])
plt.hist(X, bins=32)

## 1. Oscillating Electric Dipole

The simplest radiating source is an oscillating electric dipole.

Consider a dipole oscillating along the $\hat{\mathbf{x}}$ direction with moment $\mathbf{p}(t) = p_0 \cos(\omega\,t) \hat{\mathbf{x}}$.
Radiation is produced by accelerating charge.
In the far field $r\omega/c \gg 1$, the radiated electric field is
$$
\mathbf{E}_{\rm rad}(\mathbf{r}, t)
=
\frac{\mu_0}{4\pi r}
\left[
  \ddot{\mathbf{p}}(t_r) - \hat{\mathbf{r}}
  \left(
    \hat{\mathbf{r}}\cdot \ddot{\mathbf{p}}(t_r)
  \right)
\right],
$$
where $t_r = t - r/c$ is the delayed (retarded) time.

For the assumed dipole moment, we have $\ddot{\mathbf{p}}(t_r) = -\omega^2 p_0 \cos(\omega\,t_r) \hat{\mathbf{x}}$.
Therefore,
$$
\mathbf{E}_{\rm rad}(\mathbf{r}, t)
=
- \frac{\mu_0 \omega^2 p_0}{4\pi r} \cos(\omega\,t_r)
\left[
  \hat{\mathbf{x}} - \hat{\mathbf{r}}
  \left(
    \hat{\mathbf{r}}\cdot \hat{\mathbf{x}}
  \right)
\right].
$$

The radiated magnetic field is $\mathbf{B}_{\rm rad} = \hat{\mathbf{r}}\times\mathbf{E}_{\rm rad}/c$ but we will ignore it in this tutorial.

The important features are:
1. the field falls as $1/r$;
2. the field is transverse to the propagation direction $\hat{\mathbf{r}}$;
3. the radiation pattern is strongest perpendicular to the dipole;
4. the field vanishes along the dipole axis.